In [1]:
import os,yaml,json

In [ ]:
import os
import yaml

config_dir = '/root/wj/EZ_CLIP/configs'
dataset = ['SeAct']
config_file_templates = ['{}_train.yaml', '{}_testing.yaml']

for d in dataset:
    for config_template in config_file_templates:
        config_file = os.path.join(config_dir, d, config_template.format(d))
        print(config_file)
        # Read existing YAML content
        if os.path.exists(config_file):
            with open(config_file, 'r') as infile:
                config_data = yaml.safe_load(infile) or {}
        else:
            config_data = {}
        # basic setting
        config_data['network']['type'] = 'full_supervised' 
        config_data['weight_save_dir'] = '/root/autodl-tmp/SAMPLE'
        config_data['training_name'] = f"{config_template.format(d).replace('.yaml', '')}_full_supervised"
        config_data['network']['sim_header'] = 'Transf'
        config_data['mm_prompt']['CTX_INIT'] = 'a human action of'

        config_data['T_Adapter'] = False
        # time prompt
        config_data['prompt']['use']= True
        config_data['prompt']['DEEP']= True
        # mm prompt
        config_data['mm_prompt']['use'] = True
        config_data['mm_prompt']['N_CTX']= 2
        config_data['mm_prompt']['PROMPT_DEPTH']= 9

        # Print log directory
        logdir = os.path.join(
            config_data["weight_save_dir"],
            config_data["network"]["type"],
            config_data["network"].get("arch", "unknown_arch"),
            config_data["data"].get("dataset", d),
            config_data['training_name']
        )
        print(f"{config_template.format(d)} logdir: {logdir}")
        if config_template == '{}_testing.yaml':
            config_data['pretrain'] = os.path.join(logdir.replace('_testing','_train'), 'model_best.pt')

        
        # Write back to YAML file
        with open(config_file, 'w') as outfile:
            yaml.dump(config_data, outfile, default_flow_style=False)